In [1]:
import pandas as pd
# Example: Using a public CSV from GitHub
example_url = 'https://raw.githubusercontent.com/tidyverse/ggplot2/refs/heads/main/data-raw/diamonds.csv'

# Load the data
diamond_df = pd.read_csv(example_url)

# Display the first 5 rows if loading was successful
if diamond_df is not None:
    display(diamond_df.head())


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [2]:
diamond_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53940 non-null  float64
 1   cut      53940 non-null  object 
 2   color    53940 non-null  object 
 3   clarity  53940 non-null  object 
 4   depth    53940 non-null  float64
 5   table    53940 non-null  float64
 6   price    53940 non-null  int64  
 7   x        53940 non-null  float64
 8   y        53940 non-null  float64
 9   z        53940 non-null  float64
dtypes: float64(6), int64(1), object(3)
memory usage: 4.1+ MB


In [3]:
diamond_df.describe()

,carat,depth,table,price,x,y,z
count,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000
mean,0.797940,61.749405,57.457184,3932.799722,5.731157,5.734526,3.538734
std,0.474011,1.432621,2.234491,3989.439738,1.121761,1.142135,0.705699
min,0.200000,43.000000,43.000000,326.000000,0.000000,0.000000,0.000000
25%,0.400000,61.000000,56.000000,950.000000,4.710000,4.720000,2.910000
50%,0.700000,61.800000,57.000000,2401.000000,5.700000,5.710000,3.530000
75%,1.040000,62.500000,59.000000,5324.250000,6.540000,6.540000,4.040000
max,5.010000,79.000000,95.000000,18823.000000,10.740000,58.900000,31.800000


In [4]:
print('Missing values per column:')
print(diamond_df.isnull().sum())

Missing values per column:
carat      0
cut        0
color      0
clarity    0
depth      0
table      0
price      0
x          0
y          0
z          0
dtype: int64


In [5]:
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_filtered = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    return df_filtered

# Apply the outlier removal for each numerical column identified earlier
numerical_cols = ['carat', 'depth', 'table', 'price', 'x', 'y', 'z']
diamond_df_cleaned = diamond_df.copy()

print(f"Original shape: {diamond_df_cleaned.shape}")

for col in numerical_cols:
    initial_rows = diamond_df_cleaned.shape[0]
    diamond_df_cleaned = remove_outliers_iqr(diamond_df_cleaned, col)
    removed_rows = initial_rows - diamond_df_cleaned.shape[0]
    if removed_rows > 0:
        print(f"Removed {removed_rows} outliers from '{col}' using IQR method.")

print(f"New shape after outlier removal: {diamond_df_cleaned.shape}")

display(diamond_df_cleaned.head())

Original shape: (53940, 10)
Removed 1889 outliers from 'carat' using IQR method.
Removed 2796 outliers from 'depth' using IQR method.
Removed 342 outliers from 'table' using IQR method.
Removed 2368 outliers from 'price' using IQR method.
Removed 4 outliers from 'x' using IQR method.
Removed 1 outliers from 'y' using IQR method.
Removed 8 outliers from 'z' using IQR method.
New shape after outlier removal: (46532, 10)


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75
5,0.24,Very Good,J,VVS2,62.8,57.0,336,3.94,3.96,2.48


In [6]:
from sklearn.preprocessing import OneHotEncoder

df = diamond_df_cleaned.copy().reset_index(drop=True) # Reset index here
cat_columns = ['cut', 'color', 'clarity']

# Using pd.get_dummies for reference, but the primary method will be OneHotEncoder
df_encode = pd.get_dummies(df, columns=['cut', 'color', 'clarity'],drop_first=True)

encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_data = encoder.fit_transform(df[cat_columns])
one_hot_df = pd.DataFrame(encoded_data,
                          columns=encoder.get_feature_names_out(cat_columns),
                          index=df.index) # Ensure one_hot_df has the same index as df
df_final = pd.concat([df.drop(cat_columns, axis=1), one_hot_df], axis=1)

display(df_final.head())

,carat,depth,table,price,x,y,z,cut_Good,cut_Ideal,cut_Premium,...,color_H,color_I,color_J,clarity_IF,clarity_SI1,clarity_SI2,clarity_VS1,clarity_VS2,clarity_VVS1,clarity_VVS2
0,0.23,61.5,55.0,326,3.95,3.98,2.43,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.21,59.8,61.0,326,3.89,3.84,2.31,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,0.29,62.4,58.0,334,4.20,4.23,2.63,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.31,63.3,58.0,335,4.34,4.35,2.75,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,0.24,62.8,57.0,336,3.94,3.96,2.48,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [7]:
# outlier sudah hilang
# Categorical sudah di encode
# next, langsung split

In [8]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df_final.drop('price', axis=1)
y = df_final['price']

In [9]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# set scaling untuk KNN biar adil
def df_scaling(X, y, ts):
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42)
  scaler = StandardScaler()

  X_train_scaled = scaler.fit_transform(X_train)
  X_test_scaled = scaler.transform(X_test)
  return X_train_scaled, X_test_scaled, y_train, y_test


test_sizes = [0.1, 0.2, 0.3, 0.4]
models = [1,2,3]

knn_performance_metrics = {}
rf_performance_metrics = {}
xgb_performance_metrics = {}

for mod in models:
  for ts in test_sizes:
    print(f"\n--- Splitting with test_size={ts} ---")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42)

    print(f"Shape of X_train: {X_train.shape}")
    print(f"Shape of X_test: {X_test.shape}")
    print(f"Shape of y_train: {y_train.shape}")
    print(f"Shape of y_test: {y_test.shape}")

    if mod == 1:
      print(f"\n==== INIT KNN ====")

      X_train_scaled, X_test_scaled, y_train, y_test = df_scaling(X, y, ts) # Pakai Scaling

      knn_model = KNeighborsRegressor(
          n_neighbors=5,
          weights = 'uniform',
          algorithm = 'auto',
          leaf_size = 30,
          p = 2,
          metric = 'minkowski',
          metric_params = None,
          n_jobs = None
      )
      knn_model.fit(X_train_scaled, y_train)
      y_pred = knn_model.predict(X_test_scaled)

      # Calculate performance metrics
      mae = mean_absolute_error(y_test, y_pred)
      mse = mean_squared_error(y_test, y_pred)
      rmse = np.sqrt(mse)
      r2 = r2_score(y_test, y_pred)

      # Store metrics
      knn_performance_metrics[ts] = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

      # Print metrics
      print(f"MAE: {mae:.2f}")
      print(f"MSE: {mse:.2f}")
      print(f"RMSE: {rmse:.2f}")
      print(f"R-squared: {r2:.2f}")

    elif mod == 2: # Random Forest
      print(f"\n==== INIT RANDOM FOREST ====")

      rf_model = RandomForestRegressor(
          n_estimators=100,
          criterion='squared_error',
          max_depth=None,
      )
      rf_model.fit(X_train, y_train)
      y_pred = rf_model.predict(X_test)

      # Calculate performance metrics
      mae = mean_absolute_error(y_test, y_pred)
      mse = mean_squared_error(y_test, y_pred)
      rmse = np.sqrt(mse)
      r2 = r2_score(y_test, y_pred)

      rf_performance_metrics[ts] = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

      print(f"MAE: {mae:.2f}")
      print(f"MSE: {mse:.2f}")
      print(f"RMSE: {rmse:.2f}")
      print(f"R-squared: {r2:.2f}")

    elif mod == 3: # XGBoost
      print(f"\n==== INIT XGB ====")

      xgb_model = XGBRegressor(
          n_estimators=100,
          learning_rate=0.1,
          max_depth=3,
      )

      xgb_model.fit(X_train, y_train)
      y_pred = xgb_model.predict(X_test) # Added prediction step

      #calculate performance
      mae = mean_absolute_error(y_test, y_pred)
      mse = mean_squared_error(y_test, y_pred)
      rmse = np.sqrt(mse)
      r2 = r2_score(y_test, y_pred)

      xgb_performance_metrics[ts] = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

      print(f"MAE: {mae:.2f}")
      print(f"MSE: {mse:.2f}")
      print(f"RMSE: {rmse:.2f}")
      print(f"R-squared: {r2:.2f}")


--- Splitting with test_size=0.1 ---
Shape of X_train: (41878, 23)
Shape of X_test: (4654, 23)
Shape of y_train: (41878,)
Shape of y_test: (4654,)

==== INIT KNN ====
MAE: 286.56
MSE: 242961.94
RMSE: 492.91
R-squared: 0.97

--- Splitting with test_size=0.2 ---
Shape of X_train: (37225, 23)
Shape of X_test: (9307, 23)
Shape of y_train: (37225,)
Shape of y_test: (9307,)

==== INIT KNN ====
MAE: 293.43
MSE: 260394.31
RMSE: 510.29
R-squared: 0.96

--- Splitting with test_size=0.3 ---
Shape of X_train: (32572, 23)
Shape of X_test: (13960, 23)
Shape of y_train: (32572,)
Shape of y_test: (13960,)

==== INIT KNN ====
MAE: 306.35
MSE: 284589.23
RMSE: 533.47
R-squared: 0.96

--- Splitting with test_size=0.4 ---
Shape of X_train: (27919, 23)
Shape of X_test: (18613, 23)
Shape of y_train: (27919,)
Shape of y_test: (18613,)

==== INIT KNN ====
MAE: 314.08
MSE: 306175.12
RMSE: 553.33
R-squared: 0.95

--- Splitting with test_size=0.1 ---
Shape of X_train: (41878, 23)
Shape of X_test: (4654, 23)
Shap

In [10]:
import joblib

# Combine all performance metrics
all_performance_metrics = {
    'KNN': knn_performance_metrics,
    'RandomForest': rf_performance_metrics,
    'XGBoost': xgb_performance_metrics
}

best_r2_overall = -1
best_ts_overall = None
best_model_name = None
best_metrics_overall = None

for model_name, metrics_dict in all_performance_metrics.items():
    for ts, metrics in metrics_dict.items():
        if metrics['R2'] > best_r2_overall:
            best_r2_overall = metrics['R2']
            best_ts_overall = ts
            best_model_name = model_name
            best_metrics_overall = metrics

print(f"\nOverall Best Model: {best_model_name}")
print(f"Optimal test_size: {best_ts_overall}")
print(f"R-squared: {best_metrics_overall['R2']:.2f}")
print(f"MAE: {best_metrics_overall['MAE']:.2f}")
print(f"MSE: {best_metrics_overall['MSE']:.2f}")
print(f"RMSE: {best_metrics_overall['RMSE']:.2f}")

# Retrain the best model with the optimal test size
print(f"\nRetraining the best model: {best_model_name} with test_size={best_ts_overall}")

if best_model_name == 'KNN':
    X_train_best, X_test_best, y_train_best, y_test_best = df_scaling(X, y, best_ts_overall)
    final_best_model = KNeighborsRegressor(
        n_neighbors=5,
        weights = 'uniform',
        algorithm = 'auto',
        leaf_size = 30,
        p = 2,
        metric = 'minkowski',
        metric_params = None,
        n_jobs = None
    )
    final_best_model.fit(X_train_best, y_train_best)
    # Predict and re-evaluate to confirm metrics if needed, though not strictly required for saving
    # y_pred_best = final_best_model.predict(X_test_best)
    # confirmed_r2 = r2_score(y_test_best, y_pred_best)

elif best_model_name == 'RandomForest':
    X_train_best, X_test_best, y_train_best, y_test_best = train_test_split(X, y, test_size=best_ts_overall, random_state=42)
    final_best_model = RandomForestRegressor(
        n_estimators=100,
        criterion='squared_error',
        max_depth=None,
    )
    final_best_model.fit(X_train_best, y_train_best)

elif best_model_name == 'XGBoost':
    X_train_best, X_test_best, y_train_best, y_test_best = train_test_split(X, y, test_size=best_ts_overall, random_state=42)
    final_best_model = XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
    )
    final_best_model.fit(X_train_best, y_train_best)

# Save the best performing model
model_filename = f'best_{best_model_name.lower()}_model.pkl'
joblib.dump(final_best_model, model_filename)

print(f"Best model saved as {model_filename}")



Overall Best Model: RandomForest
Optimal test_size: 0.1
R-squared: 0.98
MAE: 200.20
MSE: 131219.48
RMSE: 362.24

Retraining the best model: RandomForest with test_size=0.1
Best model saved as best_randomforest_model.pkl
